In [ ]:
# Step 1: Install dependencies & set Hugging Face token
%pip install -q "datasets>=2.19.0" "huggingface_hub>=0.24" "transformers>=4.40.0" "torch>=2.3.0" "accelerate>=0.29.0"
import os
import getpass

# 直接设置Hugging Face token，跳过登录界面
hf_token = getpass.getpass("Paste your Hugging Face token: ")
os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGINGFACE_HUB_TOKEN'] = hf_token

print("Hugging Face token set successfully!")

In [ ]:
# Step 2: Load FLARE-FPB test set and normalize labels
from datasets import load_dataset, Dataset

LABELS = ["negative", "neutral", "positive"]

ds_raw = load_dataset("TheFinAI/flare-fpb", split="test")
print("Loaded flare-fpb test:", len(ds_raw), "columns:", ds_raw.column_names)

_alias = {"pos": "positive", "neg": "negative", "neu": "neutral",
          "bullish": "positive", "bearish": "negative"}

def _norm_label(v):
    if v is None: 
        return None
    if isinstance(v, (int, float)) or (isinstance(v, str) and v.isdigit()):
        i = int(v)
        return LABELS[i] if 0 <= i < len(LABELS) else None
    s = str(v).strip().lower()
    s = _alias.get(s, s)
    return s if s in LABELS else None

def _map_row(x):
    text = x.get("text") or x.get("sentence") or x.get("content") or x.get("input") or ""
    lab = _norm_label(x.get("label", x.get("labels", x.get("answer"))))
    return {"text": text, "choices": LABELS, "answer": lab}

ds = Dataset.from_list([{**r, **_map_row(r)} for r in ds_raw])
bad = [i for i, r in enumerate(ds) if r["answer"] not in LABELS]
print("Samples with unusable label:", len(bad))
assert len(bad) == 0, "Found unparseable labels; please check the field mapping."

In [ ]:
# Step 3: Install dependencies and record experiment metadata
%pip install -q "pandas>=2.2.2" "tqdm>=4.66.4" "scikit-learn>=1.4.0" "bitsandbytes>=0.43.0"

import os, json, time, platform, torch
from importlib.metadata import version, PackageNotFoundError

# Qwen2.5-32B-Instruct配置
MODEL_NAME = "Qwen/Qwen2.5-32B-Instruct"
MODEL_SHORT = "qwen2.5-32b-instruct"

# 检查GPU可用性
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    gpu_count = torch.cuda.device_count()
    print(f"Number of GPUs: {gpu_count}")
    for i in range(gpu_count):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"CUDA version: {torch.version.cuda}")

# 适配文件命名
run_tag = f"flare_fpb_{MODEL_SHORT.replace('-', '_').replace('/', '_').replace('.', '_')}"
save_dir = "/content"
pred_path = f"{save_dir}/{run_tag}_predictions.csv"
meta_path = f"{save_dir}/{run_tag}_metadata.json"

def ver(pkg: str) -> str:
    try:
        return version(pkg)
    except PackageNotFoundError:
        return "not-installed"

# 更新元数据
meta = {
    "dataset": "TheFinAI/flare-fpb",
    "split": "test",
    "labels": list(LABELS),
    "model": MODEL_NAME,
    "model_short": MODEL_SHORT,
    "provider": "Alibaba",
    "device": device,
    "gpu_count": gpu_count if device == "cuda" else 0,
    "transformers_version": ver("transformers"),
    "torch_version": ver("torch"),
    "accelerate_version": ver("accelerate"),
    "bitsandbytes_version": ver("bitsandbytes"),
    "datasets_version": ver("datasets"),
    "pandas": ver("pandas"),
    "tqdm": ver("tqdm"),
    "time_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
    "python": platform.python_version(),
    "note": "Qwen2.5-32B-Instruct model evaluation with Hugging Face Transformers (32B模型需要大内存)"
}

os.makedirs(save_dir, exist_ok=True)
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Meta saved ->", meta_path)
print("MODEL:", MODEL_NAME)
print("Device:", device)
if device == "cuda":
    print(f"Available GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# Step 4: Load model and tokenizer, then run inference
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import json, os, re, time
import pandas as pd
from tqdm import tqdm
import gc

# 清理GPU缓存
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()

print(f"Loading model: {MODEL_NAME}")
print("Note: Qwen2.5-32B-Instruct is a large model (32B parameters)")
print("Loading may take several minutes and requires significant GPU memory...")

# 配置4位量化以减少内存使用（32B模型必需）
if torch.cuda.is_available():
    # 检查可用内存
    free_memory = torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)
    free_memory_gb = free_memory / 1024**3
    
    print(f"Available GPU memory: {free_memory_gb:.1f} GB")
    
    # 对于32B模型，强制使用4位量化
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_storage=torch.uint8
    )
    print("Using 4-bit quantization for Qwen2.5-32B-Instruct")
else:
    print("Warning: Running 32B model on CPU will be very slow and require ~64GB RAM")
    bnb_config = None

# 加载tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, 
    token=os.environ["HF_TOKEN"],
    padding_side="left",  # 用于生成
    trust_remote_code=True  # Qwen模型需要这个
)
tokenizer.pad_token = tokenizer.eos_token

# 加载模型
print("Loading model (this will take a while for 32B model)...")
model_kwargs = {
    "torch_dtype": torch.float16 if torch.cuda.is_available() else torch.float32,
    "device_map": "auto",  # 自动分配到所有可用GPU
    "token": os.environ["HF_TOKEN"],
    "trust_remote_code": True,  # Qwen模型需要
    "attn_implementation": "flash_attention_2" if torch.cuda.is_available() else None  # 使用Flash Attention加速
}

if bnb_config:
    model_kwargs["quantization_config"] = bnb_config

try:
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)
    print("Model loaded successfully!")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Trying with lower memory configuration...")
    
    # 如果加载失败，尝试更激进的量化
    if torch.cuda.is_available():
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,  # 使用bfloat16节省更多内存
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_storage=torch.uint8
        )
        model_kwargs["quantization_config"] = bnb_config
        model_kwargs["torch_dtype"] = torch.bfloat16
    
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)
    print("Model loaded with aggressive quantization!")

model.eval()

def _strip_code_fences(s: str) -> str:
    s = s.strip()
    if s.startswith("```"):
        s = re.sub(r"^```[a-zA-Z0-9_-]*\s*", "", s)
        s = re.sub(r"\s*```$", "", s)
    return s.strip()

def _make_chat_template(text: str, choices=("negative", "neutral", "positive")):
    """为Qwen2.5创建对话格式"""
    # Qwen2.5使用标准的ChatML格式
    system_message = """You are a helpful assistant for financial sentiment analysis. Your task is to classify financial text into sentiment categories. Return ONLY a JSON object with the label."""
    
    user_message = f"""Task: classify the sentence into exactly one of these labels: {', '.join(choices)}.

Sentence: {text}

Return ONLY a JSON object on a single line, exactly in this form:
{{"label":"negative|neutral|positive"}}
No code fences, no extra text, no explanation."""
    
    # Qwen2.5对话模板 (ChatML格式)
    prompt = f"""<|im_start|>system
{system_message}<|im_end|>
<|im_start|>user
{user_message}<|im_end|>
<|im_start|>assistant
"""
    return prompt

def ask_qwen_once(text, choices=("negative", "neutral", "positive"), max_new_tokens=200):
    """使用Qwen2.5-32B-Instruct进行推理"""
    prompt = _make_chat_template(text, choices)
    
    try:
        # 编码输入
        inputs = tokenizer(
            prompt, 
            return_tensors="pt", 
            truncation=True, 
            max_length=32768,  # Qwen2.5支持32K上下文
            padding=True
        )
        
        # 注意：对于量化模型，输入已经在正确的设备上
        if bnb_config is None:  # 如果不是量化模型，手动移动
            inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # 生成输出
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.1,  # 低温度以获得确定性的输出
                do_sample=True,
                top_p=0.95,
                top_k=40,
                repetition_penalty=1.05,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                no_repeat_ngram_size=3,
                use_cache=True
            )
        
        # 解码输出
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # 提取助手回复部分
        if "<|im_start|>assistant" in generated_text:
            response = generated_text.split("<|im_start|>assistant")[-1].strip()
            # 移除可能的结束标记
            response = response.replace("<|im_end|>", "").strip()
        else:
            # 尝试其他方式提取
            response = generated_text[len(prompt):].strip()
        
        # 清理响应
        response = _strip_code_fences(response)
        
        # 解析JSON
        try:
            # 尝试直接解析
            obj = json.loads(response)
        except json.JSONDecodeError:
            # 尝试提取JSON对象
            match = re.search(r'\{.*?\}', response, re.DOTALL)
            if match:
                try:
                    obj = json.loads(match.group())
                except:
                    # 尝试修复常见的JSON格式问题
                    cleaned = match.group().replace("'", '"')
                    obj = json.loads(cleaned)
            else:
                # 尝试查找label
                for label in choices:
                    if f'"label":"{label}"' in response or f"'label':'{label}'" in response:
                        return label
                raise RuntimeError(f"Could not parse JSON from response: {response[:200]}")
        
        lab = obj.get("label")
        
        if lab not in choices:
            raise RuntimeError(f"Invalid label {lab!r}; raw json: {obj}")
        
        return lab
        
    except Exception as e:
        raise RuntimeError(f"Qwen inference error: {str(e)}")

def ask_qwen(text, choices=("negative", "neutral", "positive")):
    """带重试机制的Qwen调用"""
    for max_tokens in (200, 300, 400):
        delay = 3.0  # 32B模型需要更长延迟
        for attempt in range(4):
            try:
                return ask_qwen_once(text, choices, max_new_tokens=max_tokens)
            except RuntimeError as e:
                msg = str(e).lower()
                
                # 处理内存不足错误
                if "memory" in msg or "cuda" in msg or "out of memory" in msg:
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                        gc.collect()
                    time.sleep(delay)
                    delay = min(delay * 2, 90)  # 更长延迟
                    continue
                # 处理JSON解析错误
                if "json" in msg or "parse" in msg:
                    time.sleep(delay)
                    delay = min(delay * 1.5, 45)
                    continue
                # 处理生成错误
                if "generation" in msg or "inference" in msg:
                    time.sleep(delay)
                    delay = min(delay * 2, 60)
                    continue
                raise
            except Exception as e:
                time.sleep(delay)
                delay = min(delay * 2, 120)  # 最大120秒延迟
                if attempt < 3:
                    continue
                raise
    raise RuntimeError("Exhausted retries for this sample.")

run_tag = f"flare_fpb_{MODEL_SHORT.replace('-', '_').replace('/', '_').replace('.', '_')}"
save_dir = "/content"
pred_path = f"{save_dir}/{run_tag}_predictions.csv"
err_path = f"{save_dir}/{run_tag}_errors.csv"

rows_done = []
done_idx = set()
if os.path.exists(pred_path):
    old = pd.read_csv(pred_path)
    if "row_idx" in old.columns:
        rows_done = old.to_dict("records")
        done_idx = set(old["row_idx"].tolist())
        print(f"[resume] loaded {len(done_idx)} completed rows.")

err_rows = []
buf = []
save_every = 5  # 更频繁保存，因为32B模型处理较慢

total = len(ds)
print(f"Starting Qwen2.5-32B-Instruct model evaluation on {total} samples...")
print("Warning: 32B模型处理速度较慢，请耐心等待...")

# 定期清理GPU缓存
def cleanup_gpu():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()

start_time = time.time()
batch_start_time = start_time

for i in tqdm(range(total)):
    if i in done_idx:
        continue
    
    # 每10个样本清理一次GPU并报告进度
    if i % 10 == 0 and i > 0:
        cleanup_gpu()
        elapsed = time.time() - batch_start_time
        samples_per_sec = 10 / elapsed if elapsed > 0 else 0
        total_elapsed = time.time() - start_time
        eta = (total - i) / (i / total_elapsed) if i > 0 else 0
        
        print(f"[progress] {i}/{total} samples processed ({samples_per_sec:.3f} samples/sec)")
        print(f"[ETA] {eta/60:.1f} minutes remaining")
        batch_start_time = time.time()
    
    x = ds[i]
    text = x["text"]
    gold = x["answer"]

    try:
        pred = ask_qwen(text, LABELS)
        raw = json.dumps({"label": pred})
    except Exception as e:
        pred = "UNKNOWN"
        raw = f"ERROR: {type(e).__name__}: {e}"
        err_rows.append({"row_idx": i, "id": x.get("id", i), "error": raw, "text": text})

    buf.append({
        "row_idx": i,
        "id": x.get("id", i),
        "text": text,
        "pred_raw": raw,
        "pred": pred,
        "label": gold
    })

    if len(buf) % save_every == 0:
        out = pd.DataFrame(rows_done + buf).sort_values("row_idx")
        out.to_csv(pred_path, index=False)
        if err_rows:
            pd.DataFrame(err_rows).to_csv(err_path, index=False)
        print(f"[checkpoint] saved {len(out)}/{total} -> {pred_path}")

# 最终保存
cleanup_gpu()
out = pd.DataFrame(rows_done + buf).sort_values("row_idx")
out.to_csv(pred_path, index=False)
if err_rows:
    pd.DataFrame(err_rows).to_csv(err_path, index=False)

total_time = time.time() - start_time
print(f"[done] Qwen2.5-32B-Instruct evaluation completed in {total_time:.2f} seconds ({total_time/60:.1f} minutes)")
print(f"Average speed: {total/total_time:.3f} samples/sec")
if os.path.exists(err_path):
    err_count = len(pd.read_csv(err_path)) if os.path.getsize(err_path) > 0 else 0
    print(f"[errors] {err_count} errors logged -> {err_path}")

In [ ]:
# Step 5: Compute evaluation metrics
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import json
import time

# 加载预测结果
df = pd.read_csv(pred_path).sort_values("row_idx").drop_duplicates("row_idx", keep="last")
ok = df[df["pred"] != "UNKNOWN"].copy()

print(f"Qwen2.5-32B-Instruct Model Evaluation Results:")
print(f"Total samples: {len(df)}")
print(f"Successful predictions: {len(ok)}")
print(f"Failed predictions: {len(df) - len(ok)}")

if len(ok) > 0:
    # 计算评估指标
    f1_macro = f1_score(ok["label"], ok["pred"], labels=LABELS, average="macro", zero_division=0)
    f1_micro = f1_score(ok["label"], ok["pred"], labels=LABELS, average="micro", zero_division=0)
    f1_weighted = f1_score(ok["label"], ok["pred"], labels=LABELS, average="weighted", zero_division=0)
    accuracy = accuracy_score(ok["label"], ok["pred"])
    
    print("\n" + "="*50)
    print("EVALUATION RESULTS - Qwen2.5-32B-Instruct")
    print("="*50)
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"F1-Macro:  {f1_macro:.4f}")
    print(f"F1-Micro:  {f1_micro:.4f}")
    print(f"F1-Weighted: {f1_weighted:.4f}")
    
    # 详细分类报告
    print("\nDetailed Classification Report:")
    print(classification_report(ok["label"], ok["pred"], labels=LABELS, zero_division=0))
    
    # 混淆矩阵
    print("Confusion Matrix:")
    cm = confusion_matrix(ok["label"], ok["pred"], labels=LABELS)
    cm_df = pd.DataFrame(cm, index=LABELS, columns=LABELS)
    print(cm_df)
    
    # 保存评估结果
    eval_results = {
        "model": MODEL_NAME,
        "model_short": MODEL_SHORT,
        "dataset": "TheFinAI/flare-fpb",
        "split": "test",
        "device": device,
        "gpu_count": gpu_count if device == "cuda" else 0,
        "quantization": "4bit" if bnb_config else "none",
        "flash_attention": "enabled" if torch.cuda.is_available() else "disabled",
        "total_samples": len(df),
        "successful_predictions": len(ok),
        "failed_predictions": len(df) - len(ok),
        "accuracy": float(accuracy),
        "f1_macro": float(f1_macro),
        "f1_micro": float(f1_micro),
        "f1_weighted": float(f1_weighted),
        "total_time_seconds": float(total_time),
        "avg_samples_per_second": float(total/total_time) if total_time > 0 else 0,
        "evaluation_time": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
        "confusion_matrix": cm.tolist(),
        "labels": LABELS
    }
    
    eval_path = f"{save_dir}/{run_tag}_evaluation_results.json"
    with open(eval_path, "w") as f:
        json.dump(eval_results, f, indent=2)
    print(f"\nEvaluation results saved -> {eval_path}")
    
else:
    print("No successful predictions to evaluate!")